In [1]:
# autoreload
%load_ext autoreload
%autoreload 2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import gc
import numpy as np
import gradio as gr
import json 
import re
import subprocess
import IPython.display as ipd
import torch
import torchaudio

from einops import rearrange
from safetensors.torch import load_file
from torch.nn import functional as F
from torchaudio import transforms as T

from stable_audio_tools.interface.aeiou import audio_spectrogram_image
from stable_audio_tools.inference.generation import generate_diffusion_cond, generate_diffusion_cond_inpaint, generate_diffusion_uncond
from stable_audio_tools.models.factory import create_model_from_config
from stable_audio_tools.models.pretrained import get_pretrained_model
from stable_audio_tools.models.utils import copy_state_dict, load_ckpt_state_dict
from stable_audio_tools.inference.utils import prepare_audio
from stable_audio_tools.loraw.network import LoRAMerger, create_lora_from_config

from stable_audio_tools.interface.interfaces.diffusion_cond import create_diffusion_cond_ui

model = None
model_type = None
sample_rate = 32000
sample_size = 1920000

def load_model(model_config=None, model_ckpt_path=None, pretrained_name=None, pretransform_ckpt_path=None, device="cuda", model_half=False):
    global model, sample_rate, sample_size
    
    if pretrained_name is not None:
        print(f"Loading pretrained model {pretrained_name}")
        model, model_config = get_pretrained_model(pretrained_name)

    elif model_config is not None and model_ckpt_path is not None:
        print(f"Creating model from config")
        model = create_model_from_config(model_config)

        print(f"Loading model checkpoint from {model_ckpt_path}")
        # Load checkpoint
        copy_state_dict(model, load_ckpt_state_dict(model_ckpt_path))
        #model.load_state_dict(load_ckpt_state_dict(model_ckpt_path))

    sample_rate = model_config["sample_rate"]
    sample_size = model_config["sample_size"]

    if pretransform_ckpt_path is not None:
        print(f"Loading pretransform checkpoint from {pretransform_ckpt_path}")
        model.pretransform.load_state_dict(load_ckpt_state_dict(pretransform_ckpt_path), strict=False)
        print(f"Done loading pretransform")

    model.to(device).eval().requires_grad_(False)

    if model_half:
        model.to(torch.float16)
    
    lora = create_lora_from_config(model_config, model)
    return lora, model, model_config


model_config_path="/home/zachary/code/stable-audio-tools/stable_audio_tools/configs/model_configs/txt2audio/sao_short_inpaint.json"
ckpt_path="/home/zachary/.cache/huggingface/hub/models--stabilityai--stable-audio-open-1.0/snapshots/f21265c1e2710b3bd2386596943f0007f55f802e/model.safetensors"

if model_config_path is not None:
        # Load config from json file
    with open(model_config_path) as f:
        model_config = json.load(f)
else:
    model_config = None

device = "cuda" if torch.cuda.is_available() else "cpu"

lora, sao, conf = load_model(model_config, ckpt_path,  device=device)

/home/zachary/miniconda3/envs/sat/lib/python3.10/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


Creating model from config


/home/zachary/miniconda3/envs/sat/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading model checkpoint from /home/zachary/.cache/huggingface/hub/models--stabilityai--stable-audio-open-1.0/snapshots/f21265c1e2710b3bd2386596943f0007f55f802e/model.safetensors
Full-finetune list: ['to_input_add_embed']
Found 175 candidates for LoRA replacement
LoRA module added: model/model/to_timestep_embed/0
LoRA module added: model/model/to_timestep_embed/2
LoRA module added: model/model/to_cond_embed/0
LoRA module added: model/model/to_cond_embed/2
Full-finetune module added: model/model/to_input_add_embed
LoRA module added: model/model/transformer/layers/0/self_attn/to_qkv
LoRA module added: model/model/transformer/layers/0/self_attn/to_out
LoRA module added: model/model/transformer/layers/0/cross_attn/to_q
LoRA module added: model/model/transformer/layers/0/cross_attn/to_kv
LoRA module added: model/model/transformer/layers/0/cross_attn/to_out
LoRA module added: model/model/transformer/layers/0/ff/ff/0/proj
LoRA module added: model/model/transformer/layers/0/ff/ff/2
LoRA module

In [2]:
import torch

lora_state_d = torch.load("/home/zachary/checkpoints/s2s/ossl2_experiments/sok583sy/checkpoints/epoch=36-step=80000.ckpt", map_location="cpu")
lora.load_weights(lora_state_d)
lora.activate()

Injected 174 LoRA modules into model


In [3]:
ref_audio, sr = torchaudio.load("/home/zachary/code/stable-audio-tools/notebooks/demo_cfg_4_320962_9da78d7d47814a3c52fa.wav")
ref_audio = ref_audio[:, 524288:524288*2]

In [ ]:
# play reference audio
ipd.display(ipd.Audio(ref_audio.cpu().numpy(), rate=sr))

In [6]:
# encode reference audio
ref_audio_prepared = prepare_audio(ref_audio, sr, sample_rate, sample_size, 2, device=device)
with torch.no_grad():
    ref_latents = sao.pretransform.encode(ref_audio_prepared)


# ref_latents[..., :sample_size//2] = 0

In [ ]:
# save info mean and scale tensors for silence
mean_silence = info['mean']
scale_silence = info['scale']
# save tensors
torch.save(mean_silence, "mean_silence.pt")
torch.save(scale_silence, "scale_silence.pt")

In [ ]:
# decode to check
with torch.no_grad():
    recon_audio = sao.pretransform.decode(ref_latents)
ipd.display(ipd.Audio(recon_audio[0].cpu().numpy(), rate=sample_rate))

In [8]:
sao = sao.to(device)
# sao.model.model = torch.compile(sao.model.model)

In [ ]:
from stable_audio_tools.inference.generation import generate_diffusion_cond_blockar
from stable_audio_tools.models.inpainting import random_inpaint_mask

sample_rate = model_config["sample_rate"]
sample_size = model_config["sample_size"]




# Set up text and timing conditioning
conditioning = [{
    "prompt": "140 bpm tech house drum loop",
    "seconds_start": 0, 
    "seconds_total": 12
}]

# mask latents
inpaint_masked_input, inpaint_mask = random_inpaint_mask(ref_latents, torch.ones_like(ref_latents), **model_config['training']['inpainting']['mask_kwargs'])


# conditioning_tensors = model.conditioner(conditioning, device)
# conditioning_tensors['inpaint_mask'] = [inpaint_mask]
# conditioning_tensors['inpaint_masked_input'] = [inpaint_masked_input]

output = generate_diffusion_cond_blockar(
    sao,
    steps=50,
    cfg_scale=7,
    conditioning=conditioning,
    sample_size=sample_size,
    init_audio=(sr, ref_audio), # turn this on if you want ~10 seconds of initial audio to condition on
    sigma_min=0,
    sigma_max=1,
    sampler_type="v-ddim",
    device=device,
    ar_style='outpaint',
    block_size=98304,
    generation_length=983040
)

output = rearrange(output, "b d n -> d (b n)")
ipd.Audio(output.cpu().numpy(), rate=sample_rate)

In [ ]:
# convert to spectrogram and look at it
# do this manually dawg using torchaudio.transforms.MelSpectrogram
mel_spec_transform = T.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=1024,
    hop_length=256,
    win_length=1024,
    f_min=0,
    f_max=sample_rate//2,
    pad=0,
    n_mels=80,
    power=2.0,
    normalized=False,
)
mel_spec = mel_spec_transform(torch.tensor(output).unsqueeze(0).cpu().mean(dim=1))
mel_spec_db = T.AmplitudeToDB(top_db=80)(mel_spec)
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.imshow(mel_spec_db.squeeze().cpu().numpy(), aspect='auto', origin='lower')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.xlabel('Time (s)')
# change x label to reflext time in seconds at 44.1kHz
plt.xticks(ticks=np.arange(0, mel_spec_db.shape[-1], step=mel_spec_db.shape[-1]//10), labels=[f"{(i * 256) / sample_rate:.1f}" for i in np.arange(0, mel_spec_db.shape[-1], step=mel_spec_db.shape[-1]//10)])
plt.ylabel('Mel Frequency')
plt.tight_layout()
plt.show()